# Street View CLIP

Data Source: Dr. Stephen's Greater London Street View images.

## 1. Setup


In [ ]:
!pip -q install geopandas pyogrio rtree pyarrow transformers accelerate pillow tqdm scikit-learn


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from io import BytesIO
import zipfile
import shutil
import json

import numpy as np
import pandas as pd
import geopandas as gpd

from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
from transformers import AutoProcessor, CLIPModel
from sklearn.neighbors import NearestNeighbors

ImageFile.LOAD_TRUNCATED_IMAGES = True
pd.set_option("display.max_columns", 120)


## 2. Paths

The zip is copied to `/content` before inference to avoid slow reads from Drive.


In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/GEOG0105")

SV_DIR = BASE_DIR / "Raw Data" / "Street_view"
SHP_DIR = SV_DIR / "shp"
ZIP_PATH_DRIVE = SV_DIR / "Streetviews.zip"

OUT_DIR = BASE_DIR / "Outputs"
TABLE_DIR = OUT_DIR / "tables"
EMB_DIR = OUT_DIR / "embeddings"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_DIR = Path("/content/streetview")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH_LOCAL = LOCAL_DIR / "Streetviews.zip"

INVENTORY_PATH = TABLE_DIR / "streetview_image_inventory.csv"

CHUNK_DIR = EMB_DIR / "streetview_clip_image_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

STREETCLIP_IMAGE_OUT = EMB_DIR / "streetview_clip_image_embeddings.parquet"
STREETCLIP_SAMPLE_OUT = EMB_DIR / "streetview_clip_dino_sample_embeddings.parquet"
COVERAGE_OUT = TABLE_DIR / "streetview_clip_dino_sample_coverage.csv"
SUMMARY_OUT = TABLE_DIR / "streetview_clip_summary.json"


In [ ]:
# Copy Street View zip to local disk

if not ZIP_PATH_LOCAL.exists():
    shutil.copy2(ZIP_PATH_DRIVE, ZIP_PATH_LOCAL)


## 3. Build image list from zip

The image ID is taken from the filename stem.


In [ ]:
with zipfile.ZipFile(ZIP_PATH_LOCAL, "r") as zf:
    zip_names = zf.namelist()

image_names = [
    n for n in zip_names
    if n.lower().endswith((".jpg", ".jpeg", ".png"))
    and "__macosx" not in n.lower()
]

image_df = pd.DataFrame({"image_in_zip": image_names})
image_df["image_filename"] = image_df["image_in_zip"].apply(lambda x: Path(x).name)
image_df["image_id"] = image_df["image_filename"].apply(lambda x: Path(x).stem).astype(str)

image_df.head()


## 4. Read street-network shapefile


In [ ]:
zip_candidates = sorted(SHP_DIR.glob("*.zip"))

if len(list(SHP_DIR.glob("*.shp"))) == 0 and len(zip_candidates) > 0:
    with zipfile.ZipFile(zip_candidates[0], "r") as zf:
        zf.extractall(SHP_DIR)

shp_files = sorted(SHP_DIR.glob("*.shp"))
shp_path = shp_files[0]

street_gdf = gpd.read_file(shp_path)

if street_gdf.crs is None:
    street_gdf = street_gdf.set_crs("EPSG:27700")
elif street_gdf.crs.to_string() != "EPSG:27700":
    street_gdf = street_gdf.to_crs("EPSG:27700")

street_gdf.head()


## 5. Match shapefile ImageID to images

The ImageID column is selected by overlap with image filenames.


In [ ]:
image_id_set = set(image_df["image_id"].astype(str))

candidate_rows = []

for c in street_gdf.columns:
    if c == "geometry":
        continue

    vals = street_gdf[c].dropna().astype(str)
    if len(vals) == 0:
        continue

    overlap = len(set(vals.head(10000)).intersection(image_id_set))
    candidate_rows.append({"column": c, "overlap": overlap})

candidate_df = pd.DataFrame(candidate_rows).sort_values("overlap", ascending=False)
IMAGE_ID_COL = candidate_df.iloc[0]["column"]

sv_meta = street_gdf.copy()
sv_meta["image_id"] = sv_meta[IMAGE_ID_COL].astype(str)

if sv_meta.geometry.geom_type.isin(["LineString", "MultiLineString", "Polygon", "MultiPolygon"]).any():
    point_geom = sv_meta.geometry.representative_point()
else:
    point_geom = sv_meta.geometry

sv_meta["sv_x"] = point_geom.x
sv_meta["sv_y"] = point_geom.y

inventory = sv_meta.drop(columns="geometry").merge(
    image_df,
    on="image_id",
    how="left"
)

inventory["has_image"] = inventory["image_in_zip"].notna()
inventory.to_csv(INVENTORY_PATH, index=False)

inventory.head()


## 6. Load CLIP ViT-B/32

CLIP is used as the Street View encoder because GSV is a natural-image source.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ID = "openai/clip-vit-base-patch32"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()


## 7. Define image loading and embedding functions


In [ ]:
def load_image_from_zip(zf, image_name):
    with zf.open(image_name) as f:
        return Image.open(BytesIO(f.read())).convert("RGB")


@torch.inference_mode()
def embed_clip_images(images):
    inputs = processor(images=images, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    vision_outputs = model.vision_model(pixel_values=pixel_values)
    pooled_output = vision_outputs.pooler_output

    features = model.visual_projection(pooled_output)
    features = features / features.norm(dim=-1, keepdim=True)

    return features.detach().cpu().numpy()


def save_clip_chunk(chunk_df, out_file, batch_size=64):
    emb_batches = []
    kept_rows = []

    with zipfile.ZipFile(ZIP_PATH_LOCAL, "r") as zf:
        for start in tqdm(range(0, len(chunk_df), batch_size), leave=False):
            batch = chunk_df.iloc[start:start + batch_size]

            images = []
            rows = []

            for _, row in batch.iterrows():
                try:
                    images.append(load_image_from_zip(zf, row["image_in_zip"]))
                    rows.append(row)
                except Exception:
                    pass

            if len(images) == 0:
                continue

            emb_batches.append(embed_clip_images(images))
            kept_rows.extend(rows)

    emb = np.vstack(emb_batches)
    kept_df = pd.DataFrame(kept_rows).reset_index(drop=True)

    clip_cols = [f"streetclip_{i:03d}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=clip_cols)

    out = pd.concat(
        [
            kept_df[["image_id", "image_in_zip", "sv_x", "sv_y"]].reset_index(drop=True),
            emb_df
        ],
        axis=1
    )

    out.to_parquet(out_file, index=False)

## 8. Extract full image-level CLIP embeddings

Chunked and resumable. Existing chunks are skipped.


In [ ]:
valid_inventory = inventory[inventory["has_image"]].reset_index(drop=True)

CHUNK_SIZE = 5000
BATCH_SIZE = 64

for chunk_id, start in enumerate(range(0, len(valid_inventory), CHUNK_SIZE)):
    end = min(start + CHUNK_SIZE, len(valid_inventory))
    out_file = CHUNK_DIR / f"streetclip_image_chunk_{chunk_id:04d}.parquet"

    if out_file.exists():
        continue

    chunk_df = valid_inventory.iloc[start:end].reset_index(drop=True)
    save_clip_chunk(chunk_df, out_file, batch_size=BATCH_SIZE)


## 9. Combine image-level CLIP chunks


In [ ]:
chunk_files = sorted(CHUNK_DIR.glob("streetclip_image_chunk_*.parquet"))

streetclip_images = pd.concat(
    [pd.read_parquet(p) for p in chunk_files],
    ignore_index=True
)

streetclip_images.to_parquet(STREETCLIP_IMAGE_OUT, index=False)

streetclip_images.head()


In [ ]:
# Update Street View geometry coordinates using line midpoints

def geometry_to_point(g):
    if g.geom_type in ["LineString", "MultiLineString"]:
        return g.interpolate(0.5, normalized=True)
    elif g.geom_type in ["Polygon", "MultiPolygon"]:
        return g.representative_point()
    else:
        return g

sv_coords = street_gdf.copy()
sv_coords["image_id"] = sv_coords[IMAGE_ID_COL].astype(str)

mid_points = sv_coords.geometry.apply(geometry_to_point)

sv_coords["sv_x_new"] = mid_points.x
sv_coords["sv_y_new"] = mid_points.y

sv_coords = sv_coords[["image_id", "sv_x_new", "sv_y_new"]].drop_duplicates("image_id")

streetclip_images = streetclip_images.drop(columns=["sv_x", "sv_y"], errors="ignore")
streetclip_images = streetclip_images.merge(sv_coords, on="image_id", how="left")

streetclip_images = streetclip_images.rename(
    columns={
        "sv_x_new": "sv_x",
        "sv_y_new": "sv_y"
    }
)

streetclip_images.to_parquet(STREETCLIP_IMAGE_OUT, index=False)

streetclip_images.head()

## 10. Aggregate Street View embeddings to DINO sample points

PTAL uses a wider neighbourhood window. EPC uses a smaller local window.

- PTAL: 8 nearest images within 300 m
- EPC: 4 nearest images within 150 m
- Weight: `1 / (distance + 10)`


In [ ]:
DINO_SAMPLE_PATH = TABLE_DIR / "dino_londonwide_sample.csv"

sample = pd.read_csv(DINO_SAMPLE_PATH)
sample["sample_id"] = sample["sample_id"].astype(str)

streetclip_cols = [
    c for c in streetclip_images.columns
    if c.startswith("streetclip_")
]

sv_xy = streetclip_images[["sv_x", "sv_y"]].values
sv_emb = streetclip_images[streetclip_cols].values

nn = NearestNeighbors(n_neighbors=8, algorithm="ball_tree")
nn.fit(sv_xy)


def aggregate_for_task(task_df, k, max_dist_m):
    sample_xy = task_df[["x", "y"]].values
    distances, indices = nn.kneighbors(sample_xy, n_neighbors=k)

    meta_rows = []
    agg_embs = []

    for i in range(len(task_df)):
        d = distances[i]
        idx = indices[i]
        keep = d <= max_dist_m

        if keep.sum() == 0:
            agg = np.full(len(streetclip_cols), np.nan)
            n_images = 0
            min_dist = np.nan
            mean_dist = np.nan
        else:
            d_keep = d[keep]
            idx_keep = idx[keep]

            weights = 1 / (d_keep + 10)
            weights = weights / weights.sum()

            agg = np.average(sv_emb[idx_keep], axis=0, weights=weights)

            n_images = int(len(idx_keep))
            min_dist = float(d_keep.min())
            mean_dist = float(d_keep.mean())

        meta_rows.append({
            "sample_id": task_df.iloc[i]["sample_id"],
            "task": task_df.iloc[i]["task"],
            "sv_n_images": n_images,
            "sv_min_dist_m": min_dist,
            "sv_mean_dist_m": mean_dist,
            "sv_has_streetview": int(n_images > 0)
        })

        agg_embs.append(agg)

    meta = pd.DataFrame(meta_rows)

    emb = pd.DataFrame(
        np.vstack(agg_embs),
        columns=[f"streetclip_agg_{i:03d}" for i in range(len(streetclip_cols))]
    )

    return pd.concat([meta, emb], axis=1)


ptal_df = sample[sample["task"].eq("PTAL")].reset_index(drop=True)
epc_df = sample[sample["task"].eq("EPC")].reset_index(drop=True)

ptal_sv = aggregate_for_task(ptal_df, k=8, max_dist_m=300)
epc_sv = aggregate_for_task(epc_df, k=4, max_dist_m=150)

streetclip_sample = pd.concat([ptal_sv, epc_sv], ignore_index=True)

streetclip_sample.to_parquet(STREETCLIP_SAMPLE_OUT, index=False)

streetclip_sample.head()

## 11. Save coverage and summary


In [ ]:
coverage = (
    streetclip_sample
    .groupby("task")
    .agg(
        n_samples=("sample_id", "count"),
        n_with_streetview=("sv_has_streetview", "sum"),
        mean_n_images=("sv_n_images", "mean"),
        mean_min_dist_m=("sv_min_dist_m", "mean"),
        mean_dist_m=("sv_mean_dist_m", "mean")
    )
    .reset_index()
)

coverage["coverage_pct"] = coverage["n_with_streetview"] / coverage["n_samples"] * 100

coverage.to_csv(COVERAGE_OUT, index=False)

coverage